# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
from src.provenance import verify_checkout, verify_meridian_origin
print("Código:",verify_checkout(PROJECT_ROOT, REPO_REF))
print("Meridian:",verify_meridian_origin())
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Código: {'requested_ref': 'main', 'resolved_commit': 'ac9960a3a86e7af07a507121043c6914fcbc4195', 'dirty': True}
Meridian: {'url': 'https://github.com/google/meridian.git', 'vcs_info': {'commit_id': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'requested_revision': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'vcs': 'git'}}
Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-revision; painel: (6240, 19)


O manifesto verifica código, raw, ambiente, conclusão das etapas e hashes dos outputs; resultados incompatíveis são recalculados.

In [2]:
from src.provenance import require_stages
try:
    require_stages(range(6))
except RuntimeError as stale:
    print('Recalculando etapas anteriores:',stale)
    from src.eda import data_audit,general,temporal,relationships
    from src.geo_analysis import geo_analysis,media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit,general,temporal,geo_analysis,media_geo,relationships,official_eda][:6]:
        analysis(geo_time_panel)


## 06 — A auditoria oficial encontra os mesmos padrões?

### Pergunta
A auditoria oficial encontra os mesmos padrões?

### Motivação
Uma implementação independente pode revelar diferenças de definição, escalonamento e critérios de alerta.

### Método
Construir InputData com DataFrameInputDataBuilder; instanciar Meridian com defaults provisórios e MeridianEDA; gerar HTML, checks, warnings e artefatos numéricos. Recalcular R² ajustado na escala oficial por soma de quadrados e comparar com tolerância 1e-5.

### Interpretação antes de olhar os resultados
A API 2.1.0 gera amostras PRIOR automaticamente; não há posterior ou MCMC. Os defaults não são especificação final. INFO não significa ausência de problema; REVIEW e FAIL precisam ser lidos. R² oficial é ajustado e usa variáveis transformadas.

**Dependências:** painel validado e funções em `src/official_eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.official_eda import official_eda
report_06 = official_eda(geo_time_panel)
show(report_06,part="summary")

### Resultado observado e conclusão parcial

O relatório HTML oficial foi executado com Meridian 2.1.0, sem amostragem posterior. Os checks exportaram 16 achados: {'INFO': 10, 'REVIEW': 6}. As advertências de execução estão em meridian_warnings.csv; não foram silenciadas nem resolvidas por alteração de thresholds.

A API atual usa DataFrameInputDataBuilder. MeridianEDA gera amostras da prior no construtor para completar diagnósticos da especificação. Isso não ajusta o modelo aos dados e não produz estimativas posteriores. A especificação default é provisória: 156 knots e 8 lags. Não escolhemos hiperparâmetros finais com esta EDA.

A razão oficial é 6240/(40−1+156+2+7) = 30.59. Essa contagem é um guardrail que não conta todos os efeitos geo como parâmetros independentes. Não é prova de tamanho amostral efetivo ou de suficiência.

Os alertas de CPMU precisam de contexto: o CV máximo do custo unitário é 3.475e-08. A variação relativa minúscula é compatível com arredondamento na geração/exportação, e não evidencia dispersão econômica de custos. Não alteramos o dado ou o threshold IQR oficial para eliminar o alerta.

Uma diferença metodológica importante: os R² oficiais são ajustados e usam dados transformados; a EDA própria mostra R² brutos e por habitante não ajustados. Recalculando por soma de quadrados na mesma escala oficial, a concordância dentro de 1e-5 foi **True**. A tabela official_r2_reconciliation contém as diferenças por variável. VIF e correlações também dependem da população e agregação. Artefatos numéricos foram exportados para investigar essas diferenças.

**Revisão localizada:** 6 REVIEW têm localizações, magnitude, decisão e limite residual documentados. A severidade oficial permanece REVIEW. Correlação/VIF são reconciliados na mesma escala; IQR compara todas as chaves geo–variável–semana. CPMU tem casos nos arquivos official_cpmu_cases_geo/national. Controles/Promo nacionais oficiais usam soma por default; a EDA própria pondera por população, explicando diferenças entre escalas.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** **Revisão localizada:** 6 REVIEW têm localizações, magnitude, decisão e limite residual documentados. A severidade oficial permanece REVIEW. Correlação/VIF são reconciliados na mesma escala; IQR compara todas as chaves geo–variável–semana. CPMU tem casos nos arquivos official_cpmu_cases_geo/national. Controles/Promo nacionais oficiais usam soma por default; a EDA própria pondera por população, explicando diferenças entre escalas.
- **Quais problemas apareceram?** Guardrails oficiais extremos não são certificados de qualidade. Preservação no exemplo não certifica plausibilidade econômica.
- **O que falta investigar?** Consolidar os achados, implicações e limites de prontidão.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


### Evidências e leitura das figuras
Tabelas pequenas são exibidas integralmente. As sínteses selecionam os casos relevantes; os CSVs mantêm os dados completos.

In [4]:
show(report_06,part="evidence")

**official_checks** — 16 linhas; CSV completo em `outputs/tables/`.

,check,severity,cause,explanation
0,check_geo_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
1,check_national_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
2,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled KPI in certai...
3,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
4,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled KPI. Please c...
5,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
6,check_overall_kpi_invariability,INFO,NONE,The kpi_scaled has variability across geos and...
7,check_population_corr_raw_media,INFO,NONE,Please review the Spearman correlation between...
8,check_population_corr_scaled_treatment_control,INFO,NONE,Please review the Spearman correlation between...
9,check_geo_pairwise_corr,INFO,NONE,Please review the computed pairwise correlatio...


**data_adequacy** — 1 linhas; CSV completo em `outputs/tables/`.

,n_geos,n_times,n_knots,n_controls,n_treatments,n_data_points,n_parameters,ratio
0,40,156,156,2,7,6240,204,30.588235


**comparison** — 7 linhas; CSV completo em `outputs/tables/`.

...

,level,left,right,custom_same_scale,official,absolute_difference
0,geo,Channel0,Channel1,0.205101,0.205101,4.163336e-16
1,geo,Channel0,Channel2,0.256932,0.256932,4.996004e-16
2,geo,Channel0,Channel3,0.342039,0.342039,3.330669e-16
3,geo,Channel0,Channel4,0.188815,0.188815,1.942890e-16
4,geo,Channel0,Organic_channel0,0.345869,0.345869,0.000000e+00
5,geo,Channel0,Promo,0.367530,0.367530,9.992007e-16
6,geo,Channel0,competitor_sales_control,0.080822,0.080822,5.551115e-16
7,geo,Channel0,sentiment_score_control,0.441213,0.441213,1.609823e-15
8,geo,Channel1,Channel2,0.262364,0.262364,3.330669e-16
9,geo,Channel1,Channel3,0.320355,0.320355,7.771561e-16


**official_vif_reconciliation** — 18 linhas; CSV completo em `outputs/tables/`.

,level,variable,custom_same_scale,official,absolute_difference
0,geo,Channel0,1.458924,1.458924,0.000000e+00
1,geo,Channel1,1.228402,1.228402,2.220446e-16
2,geo,Channel2,1.719062,1.719062,2.220446e-16
3,geo,Channel3,2.456486,2.456486,4.440892e-16
4,geo,Channel4,1.773253,1.773253,4.440892e-16
5,geo,Organic_channel0,1.632197,1.632197,2.220446e-16
6,geo,Promo,1.493650,1.493650,2.220446e-16
7,geo,competitor_sales_control,2.414789,2.414789,8.881784e-16
8,geo,sentiment_score_control,2.597378,2.597378,0.000000e+00
9,national,Channel0,1.744133,1.744133,6.661338e-16


**official_thresholds** — 10 linhas; CSV completo em `outputs/tables/`.

,check_group,parameter,value,modified,scope
0,kpi_invariability_spec,std_threshold,0.0001,False,"guardrail oficial, não certificado universal d..."
1,pairwise_corr_spec,overall_threshold,0.9990,False,"guardrail oficial, não certificado universal d..."
2,pairwise_corr_spec,geo_threshold,0.9990,False,"guardrail oficial, não certificado universal d..."
3,pairwise_corr_spec,national_threshold,0.9990,False,"guardrail oficial, não certificado universal d..."
4,std_spec,geo_std_threshold,0.0001,False,"guardrail oficial, não certificado universal d..."
5,std_spec,national_std_threshold,0.0001,False,"guardrail oficial, não certificado universal d..."
6,vif_spec,geo_threshold,1000.0000,False,"guardrail oficial, não certificado universal d..."
7,vif_spec,overall_threshold,1000.0000,False,"guardrail oficial, não certificado universal d..."
8,vif_spec,national_threshold,1000.0000,False,"guardrail oficial, não certificado universal d..."
9,vif_spec,std_threshold,0.0001,False,"guardrail oficial, não certificado universal d..."


In [5]:
from IPython.display import display, FileLink
display(FileLink("outputs/reports/meridian_eda_report.html"))
# Baixe o HTML e abra no navegador para explorar os gráficos oficiais.

/workspace/scratch/5d823022ed95/tcc-revision/outputs/reports/meridian_eda_report.html